# ViT Optimizer Dynamics · Representation Geometry · Loss Landscape Lab

CIFAR-10의 작은 ViT를 같은 초기값에서 **SGD / AdamW / Prodigy / Muon**으로 50 epoch 학습합니다.

기존 중요 측정은 유지하고 더 강한 측정으로 확장합니다.

```text
성능 → gradient/update dynamics → representation geometry → class geometry
→ function-space → Hessian spectrum → mode connectivity
```

- Hessian top eigenvalue → Lanczos Ritz spectrum + min/max curvature + gradient alignment
- weight interpolation → pairwise barrier height + midpoint weight averaging
- gradient norm → gradient cosine, noise ratio, update cosine, displacement까지 확장
- PCA/UMAP 중심 → covariance spectrum, CKA, probe, Neural Collapse, margin, manifold 통계 중심

HP sweep과 scaling sweep은 제외합니다.

## 0. T4 설정

어제 결과에서 SGD는 50 epoch에 validation accuracy 약 0.7984, 100 epoch에 0.8168까지 갔고 AdamW는 최종 약 0.7766이었습니다. 그래서 이번에는 **50 epoch**, batch size 512, worker 4, AMP를 기본으로 둡니다. 비싼 Hessian은 초기점과 최종점만 계산합니다.

In [ ]:
!pip -q install datasets prodigyopt tensorboard scikit-learn
!git clone -q https://github.com/HisameOgasahara/deep-learning-diagnostics-and-improvement.git /content/dldi || git -C /content/dldi pull -q

import json, random, sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, torch
from datasets import load_dataset
from google.colab import drive
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
sys.path.insert(0, '/content/dldi/colab')
from vit_lab_model_optim import SmallViT
from vit_lab_train import train_one_optimizer
from vit_lab_repr import extract_features_and_logits, representation_diagnostics
from vit_lab_landscape import run_function_space_comparison, run_hessian_diagnostics, run_mode_connectivity

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); SEED=7; EPOCHS=50; BATCH_SIZE=512; NUM_WORKERS=4
DIAG_EPOCHS=[0,10,25,50]; DYNAMICS_EVERY=10; OPTIMIZER_NAMES=['sgd','adamw','prodigy','muon']
REP_TRAIN_SAMPLES=5000; REP_VAL_SAMPLES=2000; CONNECTIVITY_SAMPLES=1000; HESSIAN_BATCH_SIZE=64; LANCZOS_STEPS=16
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/deep_learning_diagnostics_vit'); CSV_DIR=DRIVE_ROOT/'csv'; FIG_DIR=DRIVE_ROOT/'figures'; TB_DIR=DRIVE_ROOT/'tensorboard'; SUMMARY_DIR=DRIVE_ROOT/'summaries'; LOCAL_CKPT_DIR=Path('/content/vit_optimizer_checkpoints')
for p in [CSV_DIR,FIG_DIR,TB_DIR,SUMMARY_DIR,LOCAL_CKPT_DIR]: p.mkdir(parents=True,exist_ok=True)
print(torch.__version__, DEVICE, 'native Muon=', hasattr(torch.optim,'Muon'))

## 1. 데이터와 모델

학습에는 crop/flip을 쓰고 representation 비교에는 augmentation 없는 고정 입력을 씁니다. 모델은 patch 4, embedding 192, 6 Transformer blocks, 3 heads의 ViT-Tiny 계열입니다.

In [ ]:
mean=(0.4914,0.4822,0.4465); std=(0.2470,0.2435,0.2616)
train_tf=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean,std)])
eval_tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize(mean,std)])
hf=load_dataset('uoft-cs/cifar10',split='train')
class CIFAR10HF(Dataset):
    def __init__(self,d,t): self.d=d; self.t=t
    def __len__(self): return len(self.d)
    def __getitem__(self,i): r=self.d[i]; return self.t(r['img'].convert('RGB')), int(r['label'])
aug=CIFAR10HF(hf,train_tf); ev=CIFAR10HF(hf,eval_tf)
perm=torch.randperm(len(aug),generator=torch.Generator().manual_seed(SEED)).tolist(); ti=perm[:40000]; vi=perm[40000:45000]
train_ds=Subset(aug,ti); train_eval_ds=Subset(ev,ti); val_ds=Subset(ev,vi)
kw=dict(batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),persistent_workers=True)
train_loader=DataLoader(train_ds,shuffle=True,**kw); val_loader=DataLoader(val_ds,shuffle=False,**kw)
torch.manual_seed(SEED); base=SmallViT(); INITIAL_STATE={k:v.detach().cpu().clone() for k,v in base.state_dict().items()}
print('parameters(M)=',sum(p.numel() for p in base.parameters())/1e6,'steps/epoch=',len(train_loader))

## 2. 4 optimizer 학습 + dynamics

대표 parameter에서 gradient norm/cosine, batch gradient noise ratio, update-to-weight, update cosine, parameter displacement를 기록합니다. Prodigy는 lr=1.0, Muon은 hidden 2D matrix에 적용하고 나머지는 AdamW fallback입니다.

In [ ]:
H={};D={};N={}
for name in OPTIMIZER_NAMES:
    print('\n===',name,'===')
    h,d,n=train_one_optimizer(name,INITIAL_STATE,train_loader,val_loader,DEVICE,EPOCHS,DIAG_EPOCHS,DYNAMICS_EVERY,LOCAL_CKPT_DIR,CSV_DIR,TB_DIR,amp_enabled=torch.cuda.is_available())
    H[name],D[name],N[name]=h,d,n
history=pd.concat(H.values(),ignore_index=True); history.to_csv(CSV_DIR/'all_history.csv',index=False)
plt.figure(figsize=(8,5))
for name,df in H.items(): plt.plot(df.epoch,df.val_accuracy,label=name)
plt.xlabel('epoch');plt.ylabel('validation accuracy');plt.legend();plt.grid(alpha=.2);plt.show()
final_table=history.groupby('run').tail(1)[['run','train_accuracy','val_accuracy','val_loss','seconds']].sort_values('val_accuracy',ascending=False)
final_table

## 3. Representation + class geometry

0/10/25/50 epoch에서 effective rank, covariance eigenspectrum, CKA-to-init, linear probe를 계산하고, penultimate에서는 NC1/NC2/NC3, margin, kNN purity, class radius, participation dimension, class-center correlation을 봅니다.

In [ ]:
def fixed_loader(ds,n): return DataLoader(Subset(ds,list(range(min(n,len(ds))))),batch_size=512,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),persistent_workers=True)
rep_train=fixed_loader(train_eval_ds,REP_TRAIN_SAMPLES); rep_val=fixed_loader(val_ds,REP_VAL_SAMPLES)
m=SmallViT().to(DEVICE);m.load_state_dict(INITIAL_STATE)
ITF,_,ITY=extract_features_and_logits(m,rep_train,DEVICE); IVF,IVL,IVY=extract_features_and_logits(m,rep_val,DEVICE); del m
rep_df,spec_df,class_df=representation_diagnostics(OPTIMIZER_NAMES,DIAG_EPOCHS,INITIAL_STATE,ITF,ITY,IVF,IVL,IVY,rep_train,rep_val,LOCAL_CKPT_DIR,CSV_DIR,DEVICE)
print(rep_df.head(15)); class_df

## 4. Function-space

최종 checkpoint의 logit MSE, prediction disagreement, ECE를 비교합니다. Parameter가 달라도 실제 함수가 같은지 확인하기 위한 층입니다.

In [ ]:
function_df,function_pair_df=run_function_space_comparison(OPTIMIZER_NAMES,EPOCHS,rep_val,LOCAL_CKPT_DIR,CSV_DIR,DEVICE)
print(function_df); function_pair_df

## 5. Hessian Ritz spectrum

기존 top eigenvalue 하나를 Lanczos Ritz spectrum으로 확장합니다. 최대 양의 곡률, 최소 음의 곡률 후보, spectral spread, gradient와 최대/최소 곡률 방향의 정렬을 봅니다.

In [ ]:
hl=DataLoader(Subset(val_ds,list(range(HESSIAN_BATCH_SIZE))),batch_size=HESSIAN_BATCH_SIZE,shuffle=False)
hessian_df,hessian_summary=run_hessian_diagnostics(OPTIMIZER_NAMES,EPOCHS,INITIAL_STATE,next(iter(hl)),LOCAL_CKPT_DIR,CSV_DIR,DEVICE,steps=LANCZOS_STEPS)
hessian_summary

## 6. Mode connectivity / basin

최종 checkpoint 사이의 직선 경로 $$\theta(\alpha)=(1-\alpha)\theta_A+\alpha\theta_B$$ 에서 loss barrier와 midpoint weight averaging 성능을 측정합니다.

In [ ]:
conn=fixed_loader(val_ds,CONNECTIVITY_SAMPLES)
connectivity_df,barrier_df=run_mode_connectivity(OPTIMIZER_NAMES,EPOCHS,conn,LOCAL_CKPT_DIR,CSV_DIR,DEVICE,max_samples=CONNECTIVITY_SAMPLES)
barrier_df

## 7. 최종 해석

```text
성능 차이
→ gradient/update dynamics
→ representation과 class geometry
→ function-space 차이
→ Hessian curvature
→ minimum 사이 barrier
```

이 흐름으로 optimizer가 단순히 빨리 학습했는지, 아니면 다른 표현·함수·minimum 구조를 선택했는지를 분리합니다.

In [ ]:
summary={'epochs':EPOCHS,'batch_size':BATCH_SIZE,'optimizers':OPTIMIZER_NAMES,'final_metrics':final_table.to_dict(orient='records'),'hessian':hessian_summary.to_dict(orient='records'),'mode_connectivity':barrier_df.to_dict(orient='records')}
with open(SUMMARY_DIR/'experiment_summary.json','w',encoding='utf-8') as f: json.dump(summary,f,ensure_ascii=False,indent=2)
print('saved:',DRIVE_ROOT)